# Advanced Problems: Don't Use `*args` and `**kwargs` Names Blindly

## Topic

When variable positional or keyword arguments have semantic meaning, name them clearly. Use names like `*values`, `*operands`, `**custom_attributes`, `**filters`, or `**options` instead of blindly using `*args` and `**kwargs`.

The conventional names `args` and `kwargs` are appropriate when the parameters are generic pass-through arguments, such as in decorators or wrappers. But when the arguments represent meaningful domain concepts, better names improve readability, documentation, debugging, and API design.

## Best Practices Covered

1. Use `*args` and `**kwargs` only when the arguments are truly generic.
2. Use meaningful names such as `*values`, `*records`, `**attributes`, or `**filters` when the arguments represent specific concepts.
3. Preserve function metadata when writing decorators.
4. Validate dynamic keyword arguments before applying them to objects.
5. Avoid silently accepting unknown options unless the function is explicitly designed to do so.
6. Make function signatures communicate intent.

---

# Problem 1: Refactor a Poorly Named Aggregation API

You are given a function that computes several statistics from a sequence of numbers.

The implementation works, but the names `*args` and `**kwargs` are unclear because they represent meaningful concepts.

Refactor the function so that:

- `*args` is renamed to a meaningful name.
- `**kwargs` is renamed to a meaningful name.
- The function supports the operations `sum`, `min`, `max`, and `average`.
- The keyword option `ignore_none=True` causes `None` values to be ignored.
- Unknown operations raise `ValueError`.
- Unknown keyword options raise `TypeError`.

### Starter Code

```python
def summarize(*args, **kwargs):
    pass
```

### Expected Usage

```python
summarize(1, 2, 3, operation='sum')
summarize(1, None, 3, operation='average', ignore_none=True)
```

In [1]:
def summarize(*values, **options):
    allowed_options = {'operation', 'ignore_none'}
    unknown_options = set(options) - allowed_options

    if unknown_options:
        unknown = ', '.join(sorted(unknown_options))
        raise TypeError(f'Unknown option(s): {unknown}')

    operation = options.get('operation', 'sum')
    ignore_none = options.get('ignore_none', False)

    if ignore_none:
        values = tuple(value for value in values if value is not None)

    if not values:
        raise ValueError('At least one value is required')

    if operation == 'sum':
        return sum(values)
    elif operation == 'min':
        return min(values)
    elif operation == 'max':
        return max(values)
    elif operation == 'average':
        return sum(values) / len(values)
    else:
        raise ValueError(f'Unknown operation: {operation}')

The previous cell intentionally contains a small syntax issue. Fix it before running.

The corrected version is below.

In [2]:
def summarize(*values, **options):
    allowed_options = {'operation', 'ignore_none'}
    unknown_options = set(options) - allowed_options

    if unknown_options:
        unknown = ', '.join(sorted(unknown_options))
        raise TypeError(f'Unknown option(s): {unknown}')

    operation = options.get('operation', 'sum')
    ignore_none = options.get('ignore_none', False)

    if ignore_none:
        values = tuple(value for value in values if value is not None)

    if not values:
        raise ValueError('At least one value is required')

    if operation == 'sum':
        return sum(values)
    elif operation == 'min':
        return min(values)
    elif operation == 'max':
        return max(values)
    elif operation == 'average':
        return sum(values) / len(values)
    else:
        raise ValueError(f'Unknown operation: {operation}')

In [3]:
assert summarize(1, 2, 3, operation='sum') == 6
assert summarize(1, 2, 3, operation='min') == 1
assert summarize(1, 2, 3, operation='max') == 3
assert summarize(1, 2, 3, operation='average') == 2
assert summarize(1, None, 3, operation='average', ignore_none=True) == 2

try:
    summarize(1, 2, 3, operation='median')
except ValueError as ex:
    assert 'Unknown operation' in str(ex)
else:
    raise AssertionError('Expected ValueError')

try:
    summarize(1, 2, 3, precision=2)
except TypeError as ex:
    assert 'Unknown option' in str(ex)
else:
    raise AssertionError('Expected TypeError')

print('Problem 1 tests passed.')

Problem 1 tests passed.


## Problem 1 Explanation

`*values` is better than `*args` because the positional arguments are not arbitrary. They are the numeric values being summarized.

`**options` is better than `**kwargs` because the keyword arguments are not arbitrary either. They configure the behavior of the summary operation.

Good naming makes the function self-documenting.

---

# Problem 2: Build a Safe Dynamic Model Constructor

Create a class called `Record` that accepts a required `id`, a required `name`, and additional custom fields.

Requirements:

- Use a meaningful name instead of `**kwargs`.
- Store `id` and `name` as instance attributes.
- Allow extra custom fields to be set dynamically.
- Reject custom field names that begin with `_`.
- Reject custom field names that conflict with existing attributes or methods.
- Provide a readable `__repr__`.

### Expected Usage

```python
employee = Record(1, 'Ana', department='Engineering', active=True)
employee.department
employee.active
```

In [4]:
class Record:
    def __init__(self, id, name, **custom_fields):
        self.id = id
        self.name = name

        for field_name, field_value in custom_fields.items():
            if field_name.startswith('_'):
                raise ValueError(f'Private field names are not allowed: {field_name}')

            if hasattr(self, field_name) or hasattr(type(self), field_name):
                raise ValueError(f'Field conflicts with an existing attribute: {field_name}')

            setattr(self, field_name, field_value)

    def __repr__(self):
        fields = ', '.join(f'{key}={value!r}' for key, value in vars(self).items())
        return f'Record({fields})'

In [5]:
employee = Record(1, 'Ana', department='Engineering', active=True)

assert employee.id == 1
assert employee.name == 'Ana'
assert employee.department == 'Engineering'
assert employee.active is True
assert repr(employee) == "Record(id=1, name='Ana', department='Engineering', active=True)"

try:
    Record(2, 'Ben', _token='secret')
except ValueError as ex:
    assert 'Private field names' in str(ex)
else:
    raise AssertionError('Expected ValueError')

try:
    Record(3, 'Cara', id=99)
except TypeError:
    # Python itself rejects duplicate keyword values for id.
    pass
else:
    raise AssertionError('Expected TypeError')

try:
    Record(4, 'Dan', __repr__='bad idea')
except ValueError as ex:
    assert 'Private field names' in str(ex)
else:
    raise AssertionError('Expected ValueError')

print('Problem 2 tests passed.')

Problem 2 tests passed.


## Problem 2 Explanation

`**custom_fields` communicates that extra keyword arguments become dynamic fields on the object.

The function should not blindly call `setattr` on every keyword. Dynamic attributes are powerful, but unsafe if field names are not validated.

---

# Problem 3: Design a Query Builder with Meaningful Variadic Names

Write a function called `build_query` that accepts:

- A required table name.
- Any number of selected column names.
- Any number of filters.

Requirements:

- Use `*columns`, not `*args`.
- Use `**filters`, not `**kwargs`.
- If no columns are provided, select `*`.
- Filters should be converted into a simple `WHERE` clause using equality.
- Return both the SQL string and a tuple of parameters.
- Do not directly interpolate filter values into the SQL string.

### Expected Usage

```python
build_query('users', 'id', 'email', active=True, role='admin')
```

Expected result:

```python
('SELECT id, email FROM users WHERE active = ? AND role = ?', (True, 'admin'))
```

In [6]:
def build_query(table_name, *columns, **filters):
    selected_columns = ', '.join(columns) if columns else '*'
    sql = f'SELECT {selected_columns} FROM {table_name}'

    if filters:
        conditions = [f'{field_name} = ?' for field_name in filters]
        sql += ' WHERE ' + ' AND '.join(conditions)

    parameters = tuple(filters.values())
    return sql, parameters

In [7]:
query, params = build_query('users', 'id', 'email', active=True, role='admin')

assert query == 'SELECT id, email FROM users WHERE active = ? AND role = ?'
assert params == (True, 'admin')

query, params = build_query('products', category='books')

assert query == 'SELECT * FROM products WHERE category = ?'
assert params == ('books',)

query, params = build_query('logs', 'created_at', 'message')

assert query == 'SELECT created_at, message FROM logs'
assert params == ()

print('Problem 3 tests passed.')

Problem 3 tests passed.


## Problem 3 Explanation

The positional arguments represent selected database columns, so `*columns` is clearer than `*args`.

The keyword arguments represent field filters, so `**filters` is clearer than `**kwargs`.

This also improves the readability of the body of the function:

```python
for field_name in filters
```

is much clearer than:

```python
for key in kwargs
```

---

# Problem 4: Write a Decorator Where `*args` and `**kwargs` Are Appropriate

Not every use of `*args` and `**kwargs` is bad.

Write a decorator called `trace_calls` that logs the function name and arguments before calling the wrapped function.

Requirements:

- Use `*args` and `**kwargs` because the decorator is a generic pass-through wrapper.
- Preserve the original function's metadata using `functools.wraps`.
- Return the wrapped function's result.

### Expected Usage

```python
@trace_calls
def add(x, y):
    return x + y
```

In [8]:
from functools import wraps

def trace_calls(function):
    @wraps(function)
    def inner(*args, **kwargs):
        print(f'Calling {function.__name__} with args={args}, kwargs={kwargs}')
        return function(*args, **kwargs)

    return inner

In [9]:
@trace_calls
def add(x, y):
    """Return the sum of x and y."""
    return x + y

assert add(10, 20) == 30
assert add.__name__ == 'add'
assert add.__doc__ == 'Return the sum of x and y.'

print('Problem 4 tests passed.')

Calling add with args=(10, 20), kwargs={}
Problem 4 tests passed.


## Problem 4 Explanation

This is a good use of `*args` and `**kwargs`.

The decorator does not know or care what the wrapped function's parameters mean. It is forwarding arbitrary positional and keyword arguments.

In generic pass-through code, `args` and `kwargs` are conventional and appropriate.

---

# Problem 5: Create a Strict Configuration Function

Write a function called `configure_server` that accepts:

- A required `host`.
- A required `port`.
- Additional configuration options.

Requirements:

- Use `**settings`, not `**kwargs`.
- Only allow these settings:
  - `debug`
  - `timeout`
  - `retries`
- Provide defaults:
  - `debug=False`
  - `timeout=30`
  - `retries=3`
- Reject unknown settings with a clear `TypeError`.
- Return a dictionary containing the final configuration.

### Expected Usage

```python
configure_server('localhost', 8000, debug=True, timeout=10)
```

In [10]:
def configure_server(host, port, **settings):
    default_settings = {
        'debug': False,
        'timeout': 30,
        'retries': 3,
    }

    unknown_settings = set(settings) - set(default_settings)

    if unknown_settings:
        unknown = ', '.join(sorted(unknown_settings))
        raise TypeError(f'Unknown server setting(s): {unknown}')

    final_settings = default_settings | settings

    return {
        'host': host,
        'port': port,
        **final_settings,
    }

In [11]:
config = configure_server('localhost', 8000, debug=True, timeout=10)

assert config == {
    'host': 'localhost',
    'port': 8000,
    'debug': True,
    'timeout': 10,
    'retries': 3,
}

try:
    configure_server('localhost', 8000, cache=True)
except TypeError as ex:
    assert 'Unknown server setting' in str(ex)
else:
    raise AssertionError('Expected TypeError')

print('Problem 5 tests passed.')

Problem 5 tests passed.


## Problem 5 Explanation

`**settings` is more meaningful than `**kwargs` because the keyword arguments represent configuration settings.

The function is also strict. It does not silently ignore mistakes such as:

```python
configure_server('localhost', 8000, timeuot=10)
```

A permissive `**kwargs` API might accidentally accept the misspelled option and cause confusing behavior later.

---

# Problem 6: Build a Mini Event Dispatcher

Create an `EventDispatcher` class.

It should allow event handlers to be registered and then called later.

Requirements:

- Event handlers can receive different arguments.
- The dispatcher itself should use generic `*args` and `**kwargs` when forwarding arguments to handlers.
- The public method should use meaningful names where possible.
- The `emit` method should return a list of handler results.

### Expected Usage

```python
dispatcher = EventDispatcher()

@dispatcher.on('user_created')
def send_email(username, email):
    return f'Sent email to {email}'

dispatcher.emit('user_created', 'ana', email='ana@example.com')
```

In [12]:
class EventDispatcher:
    def __init__(self):
        self._handlers_by_event = {}

    def on(self, event_name):
        def register(handler):
            self._handlers_by_event.setdefault(event_name, []).append(handler)
            return handler

        return register

    def emit(self, event_name, *args, **kwargs):
        handlers = self._handlers_by_event.get(event_name, [])
        return [handler(*args, **kwargs) for handler in handlers]

In [13]:
dispatcher = EventDispatcher()

@dispatcher.on('user_created')
def send_email(username, email):
    return f'Sent email to {email}'

@dispatcher.on('user_created')
def create_audit_log(username, email):
    return f'Created audit log for {username}'

results = dispatcher.emit('user_created', 'ana', email='ana@example.com')

assert results == [
    'Sent email to ana@example.com',
    'Created audit log for ana',
]

assert dispatcher.emit('unknown_event') == []

print('Problem 6 tests passed.')

Problem 6 tests passed.


## Problem 6 Explanation

This example uses both styles correctly.

`event_name` and `handler` are meaningful names because those parameters have clear roles.

Inside `emit`, however, `*args` and `**kwargs` are appropriate because the dispatcher does not know the signature of every possible handler. It is acting as a generic forwarding mechanism.

---

# Problem 7: Improve a Function Signature for Readability

You are given this function:

```python
def make_message(*args, **kwargs):
    sender = kwargs.get('sender', 'system')
    urgent = kwargs.get('urgent', False)
    body = ' '.join(args)
    return f'[{sender}] {body}' + ('!' if urgent else '')
```

Refactor it so that the signature better communicates intent.

Requirements:

- Rename `*args` to a meaningful name.
- Avoid `**kwargs` entirely if the accepted keyword arguments are known.
- Force `sender` and `urgent` to be keyword-only arguments.
- Preserve the original behavior.

### Expected Usage

```python
make_message('Deploy', 'complete', sender='ci', urgent=True)
```

In [14]:
def make_message(*message_parts, sender='system', urgent=False):
    body = ' '.join(message_parts)
    return f'[{sender}] {body}' + ('!' if urgent else '')

In [15]:
assert make_message('Deploy', 'complete') == '[system] Deploy complete'
assert make_message('Deploy', 'complete', sender='ci') == '[ci] Deploy complete'
assert make_message('Deploy', 'complete', sender='ci', urgent=True) == '[ci] Deploy complete!'

try:
    make_message('Deploy', 'complete', 'ci', True)
except TypeError:
    pass
else:
    raise AssertionError('Expected TypeError because sender and urgent should be keyword-only')

print('Problem 7 tests passed.')

Problem 7 tests passed.


## Problem 7 Explanation

`*message_parts` clearly describes the role of the variable positional arguments.

`sender` and `urgent` are known options, so using `**kwargs` is unnecessary. Explicit keyword-only parameters are clearer, safer, and better documented by the function signature.

---

# Final Review

## When to Use `*args` and `**kwargs`

Use them when the function is generic and does not know the meaning of the arguments:

```python
def wrapper(*args, **kwargs):
    return function(*args, **kwargs)
```

## When to Use Meaningful Names

Use meaningful names when the arguments represent domain concepts:

```python
def product(*values):
    ...

def build_query(table_name, *columns, **filters):
    ...

class Person:
    def __init__(self, name, age, **custom_attributes):
        ...
```

## Best Rule of Thumb

If you can give the variable arguments a meaningful name, you probably should.